In [1]:
import pandas as pd
from functools import reduce


In [2]:
base = r"C:\UniProj\DSLab_winter\DSLab-winter2026\deberta_processed"

paths = {
    "seed42":  base + r"\deberta_seed_42_processed\submission_seed42 (1).csv",
    "seed1337": base + r"\deberta_seed_1337_processed\submission_seed_daberta_processed1337 (1).csv",
    "seed2024": base + r"\deberta_seed_2024_processed\submission_seed_daberta_processed2024.csv",
    "noseed":  base + r"\submission_deberta_processed_noseed.csv"
}


In [3]:
dfs = {
    name: pd.read_csv(path)
    for name, path in paths.items()
}

for k, df in dfs.items():
    print(k, df.shape)


seed42 (20000, 2)
seed1337 (20000, 2)
seed2024 (20000, 2)
noseed (20000, 2)


In [4]:
df = dfs["seed42"]

for name, other in dfs.items():
    if name == "seed42":
        continue
    df = df.merge(
        other,
        on="Id",
        suffixes=("", f"_{name}")
    )


In [5]:
df = df.rename(columns={
    "Predicted": "pred_seed42",
    "Predicted_seed1337": "pred_seed1337",
    "Predicted_seed2024": "pred_seed2024",
    "Predicted_noseed": "pred_noseed"
})

df.head()


,Id,pred_seed42,pred_seed1337,pred_seed2024,pred_noseed
0,0,5,5,3,5
1,1,2,2,2,2
2,2,0,0,0,0
3,3,0,6,1,1
4,4,0,0,0,0


In [7]:
pairs = [
    ("pred_seed42", "pred_seed1337"),
    ("pred_seed42", "pred_seed2024"),
    ("pred_seed1337", "pred_seed2024"),
    ("pred_seed42", "pred_noseed"),
]

for a, b in pairs:
    agreement = (df[a] == df[b]).mean()
    print(f"{a} vs {b}: agreement = {agreement:.4f}")


pred_seed42 vs pred_seed1337: agreement = 0.8982
pred_seed42 vs pred_seed2024: agreement = 0.8966
pred_seed1337 vs pred_seed2024: agreement = 0.8955
pred_seed42 vs pred_noseed: agreement = 0.8767


In [8]:
pred_cols = ["pred_seed42", "pred_seed1337", "pred_seed2024"]

df["maj_3"] = df[pred_cols].mode(axis=1)[0]


In [9]:
df["all_equal"] = (
    (df["pred_seed42"] == df["pred_seed1337"]) &
    (df["pred_seed42"] == df["pred_seed2024"])
)

print("Unanimous:", df["all_equal"].mean().round(4))
print("Disagreement:", (1 - df["all_equal"].mean()).round(4))


Unanimous: 0.85
Disagreement: 0.1501


In [10]:
df["seed42_diff"]  = (df["pred_seed42"]  != df["maj_3"])
df["seed1337_diff"] = (df["pred_seed1337"] != df["maj_3"])
df["seed2024_diff"] = (df["pred_seed2024"] != df["maj_3"])

print("Seed42 outlier rate :", df["seed42_diff"].mean().round(4))
print("Seed1337 outlier rate:", df["seed1337_diff"].mean().round(4))
print("Seed2024 outlier rate:", df["seed2024_diff"].mean().round(4))


Seed42 outlier rate : 0.052
Seed1337 outlier rate: 0.0528
Seed2024 outlier rate: 0.0548
